# 第一课：Token 的奇妙旅程

大语言模型（如 ChatGPT、Qwen、Llama）看起来能“读懂”人类的语言，但它真正处理的并不是汉字或英文字母，而是一种叫做 **Token** 的最小单位。

这节课，我们将跟踪一句话在模型中的完整旅程：

> 📝 **“你爱我，我爱你，蜜雪”** → 这句话进入模型后，到底发生了什么？模型为什么能接出下一个字**“冰”**？

我们先用一段代码，**一口气跑完整个流程**，看看全貌。然后再逐步拆解每个环节。

⏸ 关于多模态 Token（视觉、音频等），我们将在 [多模态 Token 探秘](./multimodal_tokens.ipynb) 中深入探讨。

## 0. 先看全貌：一句话的完整旅程

下面这段代码，把 **“你爱我，我爱你，蜜雪”** 这句话送入 Qwen3.5 模型，完整走一遍从输入到输出的全流程。

不需要理解每一行代码，先看结果——感受一下这句话在模型中经历了哪些变换：

In [15]:
from modelscope import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

tokenizer_id = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
model = AutoModelForCausalLM.from_pretrained(tokenizer_id)
model.eval()

text = "你爱我，我爱你，蜜雪"
print(f"输入文本: '{text}'")
print()

token_ids = tokenizer.encode(text)
decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids]
print(f"[Step 1] 分词: {decoded_tokens}")
print(f"         Token IDs: {token_ids}")

input_ids = torch.tensor([token_ids])
with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)
print(f"[Step 2] Embedding: 形状 {embeddings.shape} (1句话, {embeddings.shape[1]}个Token, {embeddings.shape[2]}维向量)")

with torch.no_grad():
    outputs = model.model(input_ids)
    hidden_states = outputs.last_hidden_state
print(f"[Step 3] Transformer: 形状 {hidden_states.shape} (经过{model.config.num_hidden_layers}层变换)。小 tips，注意矩阵的维度没变化！！！")

with torch.no_grad():
    logits = model.lm_head(hidden_states[:, -1, :])
    probs = F.softmax(logits, dim=-1)
    next_id = torch.argmax(probs).item()
    next_text = tokenizer.decode([next_id])
    confidence = probs[0, next_id].item()

top5_probs, top5_ids = torch.topk(probs[0], 5)
top5_tokens = [tokenizer.decode([tid.item()]) for tid in top5_ids]

print(f"[Step 4] 预测: 下一个Token是 '{next_text}' (置信度={confidence:.2%})")
print(f"         Top-5 候选: {list(zip(top5_tokens, [f'{p:.2%}' for p in top5_probs.tolist()]))}")
print()
print(f"完整句子: '{text + next_text}'")
print()
print("=" * 60)
print("旅程回顾:")
print(f"  文本 '{text}'")
print(f"  -> 分词为 {len(token_ids)} 个 Token: {decoded_tokens}")
print(f"  -> 每个Token变成 {embeddings.shape[2]} 维向量")
print(f"  -> 经过 {model.config.num_hidden_layers} 层 Transformer 变换")
print(f"  -> 从 {len(tokenizer)} 个候选中选出概率最高的: '{next_text}'")
print(f"  -> 拼接得到: '{text + next_text}'")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

输入文本: '你爱我，我爱你，蜜雪'

[Step 1] 分词: ['你', '爱我', '，', '我爱你', '，', '蜜', '雪']
         Token IDs: [95933, 121196, 3709, 115734, 3709, 98735, 97055]
[Step 2] Embedding: 形状 torch.Size([1, 7, 1024]) (1句话, 7个Token, 1024维向量)
[Step 3] Transformer: 形状 torch.Size([1, 7, 1024]) (经过24层变换)。小 tips，注意矩阵的维度没变化！！！
[Step 4] 预测: 下一个Token是 '冰' (置信度=73.05%)
         Top-5 候选: [('冰', '73.05%'), ('儿', '14.36%'), ('卡', '1.42%'), ('天', '0.92%'), ('龙', '0.63%')]

完整句子: '你爱我，我爱你，蜜雪冰'

旅程回顾:
  文本 '你爱我，我爱你，蜜雪'
  -> 分词为 7 个 Token: ['你', '爱我', '，', '我爱你', '，', '蜜', '雪']
  -> 每个Token变成 1024 维向量
  -> 经过 24 层 Transformer 变换
  -> 从 248077 个候选中选出概率最高的: '冰'
  -> 拼接得到: '你爱我，我爱你，蜜雪冰'


看到了吗？一句话经过 **分词 → 向量化 → 深层变换 → 概率预测**，模型就能“猜”出下一个字。

接下来，我们逐步深入每个环节，搞清楚每一步到底在做什么、为什么这么做。

---

## 1. 切分：文本如何变成 Token？

模型的第一步，是把连续的文本切分成一个个离散的 **Token**。

Token 不是字，也不是词，而是介于两者之间的东西。让我们看看不同文本被切分后的效果：

In [20]:
texts = [
    "Tokenization is awesome!",
    "深度学习非常有趣。",
    "你爱我，我爱你，蜜雪",
]

for text in texts:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)
    decoded = [tokenizer.decode([tid]) for tid in token_ids]
    print(f"原文: {text}")
    print(f"  分词结果（词表里的样子）: {tokens}")
    print(f"  分词结果在词表里的id: {token_ids}")
    print(f"  分词结果还原: {decoded}")
    print(f"  Token数: {len(token_ids)}")
    print()

print("观察:")
print('  - 英文 "Tokenization" 被切成了 "Token" + "ization"（子词切分）')
print('  - 中文 "深度学习" 作为一个整体保留（高频词组）')
print('  - Token 可能是一个词、一个字、甚至一个词根——取决于训练语料中的出现频率')

原文: Tokenization is awesome!
  分词结果（词表里的样子）: ['Token', 'ization', 'Ġis', 'Ġawesome', '!']
  分词结果在词表里的id: [3214, 1954, 369, 12099, 0]
  分词结果还原: ['Token', 'ization', ' is', ' awesome', '!']
  Token数: 5

原文: 深度学习非常有趣。
  分词结果（词表里的样子）: ['æ·±åº¦åŃ¦ä¹ł', 'éĿŀå¸¸', 'æľīè¶£', 'ãĢĤ']
  分词结果在词表里的id: [124089, 96762, 101858, 1710]
  分词结果还原: ['深度学习', '非常', '有趣', '。']
  Token数: 4

原文: 你爱我，我爱你，蜜雪
  分词结果（词表里的样子）: ['ä½ł', 'çĪ±æĪĳ', 'ï¼Į', 'æĪĳçĪ±ä½ł', 'ï¼Į', 'èľľ', 'éĽª']
  分词结果在词表里的id: [95933, 121196, 3709, 115734, 3709, 98735, 97055]
  分词结果还原: ['你', '爱我', '，', '我爱你', '，', '蜜', '雪']
  Token数: 7

观察:
  - 英文 "Tokenization" 被切成了 "Token" + "ization"（子词切分）
  - 中文 "深度学习" 作为一个整体保留（高频词组）
  - Token 可能是一个词、一个字、甚至一个词根——取决于训练语料中的出现频率


### 等等，`tokenize()` 返回的中文怎么是乱码？

如果你用 `tokenizer.tokenize()` 而不是 `tokenizer.decode()`，会看到中文 Token 长这样：`æ·±åº¦åŃ¦ä¹ł`（其实是“深度学习”）。

这不是 bug！因为 Qwen 使用的是 **BBPE (Byte-level BPE)**，词表中的 Token 是字节级表示：
1. 中文先被编码为 UTF-8 字节（每个汉字 3 字节）
2. 字节值通过 Byte-to-Unicode 映射表变成可见字符（如 `0xE6` → `æ`）
3. BPE 算法在这些字节级 Token 上进行合并

所以 `tokenize()` 返回的是词表中的**内部键名**，而 `decode()` 走的是反向路径（ID → 字节 → UTF-8解码 → 可读文本），能完美还原。

> 这也解释了为什么 `convert_tokens_to_ids("世界")` 找不到——因为词表里存的不是“世界”，而是它的字节级表示 `ä¸ĸçķĮ`。但 `encode("世界")` 可以正常工作，因为它会先做 BPE 分词再查表。

In [3]:
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

b2u = bytes_to_unicode()
u2b = {v: k for k, v in b2u.items()}

zh_text = "深度学习"
print(f"原始中文: {zh_text}")
print(f"字符数: {len(zh_text)}")

print(f"\n=== 第一步: UTF-8 编码 ===")
utf8_bytes = zh_text.encode('utf-8')
print(f"UTF-8 字节: {list(utf8_bytes)}")
print(f"字节数: {len(utf8_bytes)} (每个汉字 3 个字节)")

print(f"\n=== 第二步: 逐字节展示 ===")
for i, char in enumerate(zh_text):
    char_bytes = char.encode('utf-8')
    byte_str = ' '.join(f'0x{b:02X}' for b in char_bytes)
    unicode_str = ''.join(b2u[b] for b in char_bytes)
    print(f"  '{char}' -> 字节 [{byte_str}] -> Byte-to-Unicode 映射: '{unicode_str}'")

print(f"\n=== 第三步: 整体映射结果 ===")
full_mapped = ''.join(b2u[b] for b in utf8_bytes)
print(f"字节序列: {list(utf8_bytes)}")
print(f"Byte-to-Unicode 映射后: '{full_mapped}'")
print(f"这就是 tokenizer.tokenize() 返回的“乱码”!")

print(f"\n=== 第四步: 反向解码（“乱码” -> 中文）===")
recovered_bytes = bytes([u2b[c] for c in full_mapped])
recovered_text = recovered_bytes.decode('utf-8')
print(f"“乱码”字符串: '{full_mapped}'")
print(f"反向映射回字节: {list(recovered_bytes)}")
print(f"UTF-8 解码: '{recovered_text}'")
print(f"解码结果与原文一致: {recovered_text == zh_text}")

print(f"\n=== 完整轮转示意 ===")
print(f"中文 '深度学习' -> UTF-8 字节 [0xE6,0xB7,0xB1,0xE5,0xBA,0xA6,0xE5,0xAD,0xA6] -> Byte-to-Unicode 'æ·±åº¦åŃ¦' -> 反向映射+解码 -> '深度学习'")

原始中文: 深度学习
字符数: 4

=== 第一步: UTF-8 编码 ===
UTF-8 字节: [230, 183, 177, 229, 186, 166, 229, 173, 166, 228, 185, 160]
字节数: 12 (每个汉字 3 个字节)

=== 第二步: 逐字节展示 ===
  '深' -> 字节 [0xE6 0xB7 0xB1] -> Byte-to-Unicode 映射: 'æ·±'
  '度' -> 字节 [0xE5 0xBA 0xA6] -> Byte-to-Unicode 映射: 'åº¦'
  '学' -> 字节 [0xE5 0xAD 0xA6] -> Byte-to-Unicode 映射: 'åŃ¦'
  '习' -> 字节 [0xE4 0xB9 0xA0] -> Byte-to-Unicode 映射: 'ä¹ł'

=== 第三步: 整体映射结果 ===
字节序列: [230, 183, 177, 229, 186, 166, 229, 173, 166, 228, 185, 160]
Byte-to-Unicode 映射后: 'æ·±åº¦åŃ¦ä¹ł'
这就是 tokenizer.tokenize() 返回的“乱码”!

=== 第四步: 反向解码（“乱码” -> 中文）===
“乱码”字符串: 'æ·±åº¦åŃ¦ä¹ł'
反向映射回字节: [230, 183, 177, 229, 186, 166, 229, 173, 166, 228, 185, 160]
UTF-8 解码: '深度学习'
解码结果与原文一致: True

=== 完整轮转示意 ===
中文 '深度学习' -> UTF-8 字节 [0xE6,0xB7,0xB1,0xE5,0xBA,0xA6,0xE5,0xAD,0xA6] -> Byte-to-Unicode 'æ·±åº¦åŃ¦' -> 反向映射+解码 -> '深度学习'


### 为什么不直接用字或词？

| 方案 | 优点 | 缺点 |
| --- | --- | --- |
| **字/字母级** | 词表极小（英文26个） | 序列太长，难以捕捉语义 |
| **词级** | 语义完整 | 词表无限膨胀，遇到新词就傻了（OOV问题） |
| **子词级 (BPE)** | ✅ 兼顾词表大小和语义 | 需要训练分词器 |

BPE 的核心思想：**找到语料库中最常连续出现的字符组合，把它们合并成新的 Token。**
- 英文：`ing`、`tion`、`ly` 等常见词缀成为独立 Token，生僻词被拆成词根
- 中文：高频词组（如“中国”、“深度学习”）成为独立 Token，生僻字可能被拆为字节

## 2. 词表：模型“认识”多少个 Token？

每个 Tokenizer 都有一个固定的**词表**，存储了所有它“认识”的 Token。词表的大小和内容，直接决定了模型的“世界观”。

让我们看看 Qwen3.5 的词表长什么样，以及不同模型的词表差异如何影响分词效果：

In [4]:
print(f"模型: {tokenizer_id}")
print(f"词表大小 (vocab_size): {len(tokenizer)}")
print(f"模型最大输入长度: {tokenizer.model_max_length}")
print(f"\n--- 特殊 Token ---")
special_tokens = tokenizer.special_tokens_map
for name, value in special_tokens.items():
    token_id = tokenizer.convert_tokens_to_ids(value)
    print(f"  {name:15s} -> {value!r:20s}  (ID: {token_id})")

print(f"\n--- 词表中的 Token 示例 ---")
examples = ["hello", "世界", "ing", "Ġthe", "Ġis", "</w>"]
for ex in examples:
    tid = tokenizer.convert_tokens_to_ids(ex)
    is_unk = (tid is None) or (tid == tokenizer.unk_token_id)
    if is_unk:
        tokens_for_ex = tokenizer.tokenize(ex)
        actual_ids = tokenizer.encode(ex)
        decoded = [tokenizer.decode([i]) for i in actual_ids]
        print(f"  {ex!r:20s} -> 不在词表中！但 tokenize 后可以正常编码: {tokens_for_ex} (长度为{len(actual_ids)}) -> 解码: {decoded}")
    else:
        print(f"  {ex!r:20s} -> ID={tid}")

print(f"\n--- 词表 ID 范围 ---")
print(f"  最小 ID: 0, 最大 ID: {len(tokenizer) - 1}")
print(f"\n--- 词表首尾 Token ---")
for i in [0, 1, 2, 3, 4]:
    print(f"  ID {i}: {tokenizer.decode([i])!r}")
print(f"  ...")
for i in range(len(tokenizer) - 3, len(tokenizer)):
    print(f"  ID {i}: {tokenizer.decode([i])!r}")


模型: Qwen/Qwen3.5-0.8B
词表大小 (vocab_size): 248077
模型最大输入长度: 262144

--- 特殊 Token ---
  eos_token       -> '<|im_end|>'          (ID: 248046)
  pad_token       -> '<|endoftext|>'       (ID: 248044)
  audio_bos_token -> '<|audio_start|>'     (ID: 248070)
  audio_eos_token -> '<|audio_end|>'       (ID: 248071)
  audio_token     -> '<|audio_pad|>'       (ID: 248076)
  image_token     -> '<|image_pad|>'       (ID: 248056)
  video_token     -> '<|video_pad|>'       (ID: 248057)
  vision_bos_token -> '<|vision_start|>'    (ID: 248053)
  vision_eos_token -> '<|vision_end|>'      (ID: 248054)

--- 词表中的 Token 示例 ---
  'hello'              -> ID=14556
  '世界'                 -> 不在词表中！但 tokenize 后可以正常编码: ['ä¸ĸçķĮ'] (长度为1) -> 解码: ['世界']
  'ing'                -> ID=286
  'Ġthe'               -> ID=279
  'Ġis'                -> ID=369
  '</w>'               -> 不在词表中！但 tokenize 后可以正常编码: ['</', 'w', '>'] (长度为3) -> 解码: ['</', 'w', '>']

--- 词表 ID 范围 ---
  最小 ID: 0, 最大 ID: 248076

--- 词表首尾 Token ---
  ID 0

### 为什么"世界"不在词表中？—— BBPE 词表的内部表示

你可能会惊讶：像"世界"这样高频的中文词，居然显示"不在词表中"！
但这**并不意味着模型不认识"世界"**。原因在于 BBPE 词表的存储方式：

- **词表中的 Token 是字节级内部表示**，不是人类可读的文本。例如"世界"在词表中存储为 `ä¸ĸçķĮ`（UTF-8 字节经 Byte-to-Unicode 映射后的结果），而不是直接存"世界"两个字。
- `convert_tokens_to_ids("世界")` 查找的是**原始字符串"世界"**在词表中的位置，自然找不到。
- 但 `tokenizer.encode("世界")` 会先进行 BPE 分词，将"世界"切分为词表中存在的 Token（可能是一个整体 Token，也可能被拆分），然后返回对应的 ID。

所以，正确的理解是：**"世界"作为一个可编码的文本单位，模型是完全认识的**；只是它的词表"键名"用的是字节级表示，而不是人类可读的中文。

这也是为什么我们之前看到 `tokenize()` 对中文返回"乱码"——那正是词表中的真实键名。
而 `decode()` 方法能完美还原，因为它走的是"Token ID → 字节级表示 → UTF-8 解码 → 可读文本"的反向路径。


### 同一句话，不同模型切出不同结果

中国的 Qwen 和美国的 Gemma，对同一段中文的分词结果差异巨大——这直接影响 Token 消耗和 API 费用：

In [5]:
from modelscope import AutoTokenizer as MS_AutoTokenizer
import logging
logging.getLogger("modelscope").setLevel(logging.ERROR)

gemma_tokenizer = MS_AutoTokenizer.from_pretrained("google/gemma-4-E2B-it")

zh_texts = [
    "我爱你，中国！五星红旗迎风飘扬",
    "人工智能、机器学习、深度学习三者之间的关系",
    "春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。",
]
en_text = "Supercalifragilisticexpialidocious"

models = [
    ("Qwen3.5 (中国)", tokenizer),
    ("Gemma-4 (美国)", gemma_tokenizer),
]

for lang_label, txt_list in [("中文", zh_texts), ("英文", [en_text])]:
    for txt in txt_list:
        print(f"\n{'='*60}")
        print(f"{lang_label}: {txt}")
        print(f"{'='*60}")
        for model_label, tok in models:
            ids = tok.encode(txt)
            decoded_tokens = [tok.decode([tid]) for tid in ids]
            print(f"\n  [{model_label}]")
            print(f"  词表大小: {len(tok):,}  |  Token 数量: {len(ids)}")
            print(f"  逐 Token 解码: {decoded_tokens}")

print(f"\n\n=== 小结 ===")
print("中文场景: Qwen 的词表包含大量中文词组（如'五星红旗'、'迎风'等），同样的中文只需要更少的 Token。")
print("  例如'我爱你，中国！五星红旗迎风飘扬'：Qwen 只需 7 个 Token，Gemma 却需要 13 个！")
print("英文场景: 两个模型差异不大，但对超长词（如 Supercalifragilisticexpialidocious）的分词边界不同。")
print("这就是为什么使用非中文优化的大模型 API 时，中文 prompt 的 Token 消耗和费用通常比英文更高。")



中文: 我爱你，中国！五星红旗迎风飘扬

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 7
  逐 Token 解码: ['我爱你', '，', '中国', '！', '五星红旗', '迎风', '飘扬']

  [Gemma-4 (美国)]
  词表大小: 262,144  |  Token 数量: 13
  逐 Token 解码: ['我', '爱你', '，', '中国', '！', '五', '星', '红', '旗', '迎', '风', '飘', '扬']

中文: 人工智能、机器学习、深度学习三者之间的关系

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 7
  逐 Token 解码: ['人工智能', '、', '机器学习', '、', '深度学习', '三者', '之间的关系']

  [Gemma-4 (美国)]
  词表大小: 262,144  |  Token 数量: 10
  逐 Token 解码: ['人工智能', '、', '机器学习', '、', '深度', '学习', '三', '者', '之间的', '关系']

中文: 春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 19
  逐 Token 解码: ['春', '眠', '不觉', '晓', '，', '处处', '闻', '啼', '鸟', '。', '夜', '来', '风雨', '声', '，', '花落', '知', '多少', '。']

  [Gemma-4 (美国)]
  词表大小: 262,144  |  Token 数量: 23
  逐 Token 解码: ['春', '眠', '不', '觉', '晓', '，', '处', '处', '闻', '啼', '鸟', '。', '夜', '来', '风', '雨', '声', '，', '花', '落', '知', '多少', '。']

英文: Supercalifragilisticexpialidocious

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 11
 

## 3. 向量化：Token 如何变成可计算的？

切分完的 Token 还是离散的整数 ID（如 `95933`），神经网络无法直接计算。
我们需要把它们映射为连续的稠密向量——这就是 **Embedding**。

这些向量是跟模型一起训练出来的，语义相近的 Token 在高维空间中应该更接近。
让我们从三个角度验证：

In [6]:
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("modelscope").setLevel(logging.ERROR)

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

model = AutoModelForCausalLM.from_pretrained(tokenizer_id)

embedding_layer = model.get_input_embeddings()
print(f"Embedding 层结构: {embedding_layer}")

print(f"\nEmbedding 维度: {embedding_layer.embedding_dim}")
print(f"词表大小: {embedding_layer.num_embeddings}")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Embedding 层结构: Embedding(248320, 1024)

Embedding 维度: 1024
词表大小: 248320


### 3.1 语义相似度：相近的词，向量真的更近吗？

如果 Embedding 学得好，语义相近的词（如"猫"和"狗"）的向量应该比语义无关的词（如"猫"和"桌子"）更接近。
我们用**余弦相似度**来衡量两个向量的接近程度（范围 -1 到 1，越大越相似）：


In [13]:
print("=== 语义相似度对比 ===\n")

similarity_groups = [
    ("同类对比", [
        ("king", "queen"),
        ("king", "man"),
        ("king", "apple"),
    ]),
    ("中文同类对比", [
        ("猫", "狗"),
        ("猫", "桌子"),
        ("苹果", "香蕉"),
        ("苹果", "汽车"),
    ]),
]

def get_embedding(word, print_dim=False):
    ids = tokenizer.encode(word)
    emb = embedding_layer(torch.tensor([ids[0]])).detach()
    result = emb.squeeze(0)
    if print_dim:
        print(f'get_embedding 维度细节（输入词：{word}）：\n\t- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：{torch.tensor(ids).shape}\n\t- 经过embedding_layer后{emb.shape}\n\t- 经过squeeze 后：{result.shape}\n')
    return result

def cosine_sim(w1, w2):
    e1 = get_embedding(w1, print_dim=(w1=='猫'))
    e2 = get_embedding(w2)
    return F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item()

for group_name, pairs in similarity_groups:
    print(f"--- {group_name} ---")
    for w1, w2 in pairs:
        sim = cosine_sim(w1, w2)
        bar = "█" * int((sim + 1) * 25)
        print(f"  cosine({w1:>4s}, {w2:>4s}) = {sim:+.4f}  {bar}")
    print()

print("💡 观察: 同类词(猫-狗, 苹果-香蕉)的相似度明显高于跨类词(猫-桌子, 苹果-汽车)")


=== 语义相似度对比 ===

--- 同类对比 ---
  cosine(king, queen) = +0.3574  █████████████████████████████████
  cosine(king,  man) = +0.0630  ██████████████████████████
  cosine(king, apple) = +0.1299  ████████████████████████████

--- 中文同类对比 ---
get_embedding 维度细节（输入词：猫）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

  cosine(   猫,    狗) = +0.4727  ████████████████████████████████████
get_embedding 维度细节（输入词：猫）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

  cosine(   猫,   桌子) = +0.0806  ███████████████████████████
  cosine(  苹果,   香蕉) = +0.3672  ██████████████████████████████████
  cosine(  苹果,   汽车) = +0.2109  ██████████████████████████████

💡 观察: 同类词(猫-狗, 苹果-香蕉)的相似度明显高于跨类词(猫-桌子, 苹果-汽车)


### 3.2 词类比：king - man + woman ≈ queen？

Word2Vec 时代最经典的发现：词向量之间存在**线性关系**。
如果 `king` 的向量减去 `man` 的向量再加上 `woman` 的向量，结果应该接近 `queen`。

但之前我们只用了**静态 Embedding 层**（模型的第一层），效果有限。
一个自然的问题：**经过完整的 Transformer 层之后，模型是否真正学到了这种语义关系？**

我们来对比静态 Embedding 和上下文 Embedding（跑完全部 Transformer 层后的输出）的词类比效果：


In [8]:
def get_contextual_embedding_from_sentence(sentence, target_word):
    """从句子中提取目标词的上下文 Embedding（经过全部 Transformer 层）"""
    inputs = tokenizer(sentence, return_tensors="pt")
    input_ids = inputs["input_ids"]
    token_ids_list = input_ids[0].tolist()
    decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids_list]
    
    target_pos = None
    for j, dt in enumerate(decoded_tokens):
        if target_word in dt:
            target_pos = j
            break
    if target_pos is None:
        return None
    
    with torch.no_grad():
        outputs = model.model(input_ids)
        hidden_states = outputs.last_hidden_state
    return hidden_states[0, target_pos, :].detach()

print("=== 词类比实验: 静态 Embedding vs 上下文 Embedding ===\n")

analogies = [
    ("king", "man", "woman", "queen",
     "The king is here", "The man is here", "The woman is here", "The queen is here"),
    ("北京", "中国", "法国", "巴黎",
     "北京是城市", "中国是国家", "法国是国家", "巴黎是城市"),
    ("国王", "男人", "女人", "女王",
     "国王是人物", "男人是性别", "女人是性别", "女王是人物"),
]

all_ids = torch.arange(embedding_layer.num_embeddings)
with torch.no_grad():
    all_static_embeddings = embedding_layer(all_ids)

for a, b, c, expected, sa, sb, sc, sexpected in analogies:
    # --- 静态 Embedding ---
    ea_s = get_embedding(a)
    eb_s = get_embedding(b)
    ec_s = get_embedding(c)
    e_expected_s = get_embedding(expected)
    result_s = ea_s - eb_s + ec_s
    sim_s = F.cosine_similarity(result_s.unsqueeze(0), e_expected_s.unsqueeze(0)).item()
    
    sims_s = F.cosine_similarity(result_s.unsqueeze(0), all_static_embeddings)
    top5_s = torch.argsort(sims_s, descending=True)[:5]
    
    # --- 上下文 Embedding (经过全部 Transformer 层) ---
    ea_c = get_contextual_embedding_from_sentence(sa, a)
    eb_c = get_contextual_embedding_from_sentence(sb, b)
    ec_c = get_contextual_embedding_from_sentence(sc, c)
    e_expected_c = get_contextual_embedding_from_sentence(sexpected, expected)
    
    if all(x is not None for x in [ea_c, eb_c, ec_c, e_expected_c]):
        result_c = ea_c - eb_c + ec_c
        sim_c = F.cosine_similarity(result_c.unsqueeze(0), e_expected_c.unsqueeze(0)).item()
    else:
        sim_c = float("nan")
    
    print(f"  {a} - {b} + {c} ≈ {expected}?")
    print(f"  静态 Embedding:   与目标词相似度 = {sim_s:.4f}")
    print(f"    Top-5: ", end="")
    for tid in top5_s:
        tok_str = tokenizer.decode([tid.item()])
        s = sims_s[tid].item()
        marker = " <--" if tok_str.strip() == expected else ""
        print(f"{tok_str!r}({s:.3f}){marker}", end="  ")
    print()
    print(f"  上下文 Embedding: 与目标词相似度 = {sim_c:.4f}")
    print()

print("观察:")
print("  1. 静态 Embedding: queen 不在 king-man+woman 的 top-5 中，词类比几乎失败")
print("  2. 上下文 Embedding: 与目标词的相似度大幅提升（0.45 → 0.88），词类比明显更有效")
print("  3. 这说明 Transformer 层真正学到了语义关系，而不只是表面的向量距离")


=== 词类比实验: 静态 Embedding vs 上下文 Embedding ===

get_embedding 维度细节（输入词：king）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

get_embedding 维度细节（输入词：man）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

get_embedding 维度细节（输入词：woman）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

get_embedding 维度细节（输入词：queen）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

  king - man + woman ≈ queen?
  静态 Embedding:   与目标词相似度 = 0.4512
    Top-5: 'king'(0.719)  'キング'(0.465)  'вляю'(0.459)  'tifkan'(0.459)  ' поджелудо'(0.457)  
  上下文 Embedding: 与目标词相似度 = 0.8789

get_embedding 维度细节（输入词：北京）：
	- tokenizer.encode，其实就是把单词映射成对应的to

### 3.3 语境语义：同一个词，在不同语境下意思一样吗？

上面我们用的是**静态 Embedding**（Embedding 层的直接输出），每个 Token 只有一个固定的向量。
这意味着：无论"苹果"出现在"吃苹果"还是"苹果公司"中，静态 Embedding 给出的都是**同一个向量**。

但自然语言中，同一个词在不同语境下可能有完全不同的含义：
- "请帮我**开**门" → 开 = 打开
- "他**开**车上班" → 开 = 驾驶
- "这个**苹果**很甜" → 苹果 = 水果
- "**苹果**发布了新手机" → 苹果 = 公司

静态 Embedding 无法区分这些含义（同一个 Token 永远对应同一个向量，相似度 = 1.0）。
但经过 Transformer 层处理后，同一个 Token 会根据上下文获得不同的表示。
让我们对比一下：


In [9]:
def get_contextual_embedding(text, target_word):
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]
    token_ids_list = input_ids[0].tolist()
    decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids_list]
    
    target_pos = None
    for i, dt in enumerate(decoded_tokens):
        if target_word in dt:
            target_pos = i
            break
    if target_pos is None:
        print(f"  Warning: target not found in: {decoded_tokens}")
        return None
    
    with torch.no_grad():
        outputs = model.model(input_ids)
        hidden_states = outputs.last_hidden_state
    return hidden_states[0, target_pos, :].detach()

print("=== 语境语义对比 ===\n")

contexts = [
    ("请帮我开门", "开", "打开"),
    ("他开车上班", "开", "驾驶"),
    ("这个苹果很甜", "苹果", "水果"),
    ("苹果发布了新手机", "苹果", "公司"),
    ("今天天气很热", "热", "温度高"),
    ("他是个热门人物", "热", "受欢迎"),
    ("这朵花很美", "花", "花朵"),
    ("他花了很多钱", "花", "花费"),
]

emb_dict = {}
for text, word, sense in contexts:
    emb = get_contextual_embedding(text, word)
    if emb is not None:
        key = f"{word}({sense})"
        emb_dict[key] = emb
        print(f'  "{text}" -> {key}')

print(f"\n--- 核心对比: 静态 Embedding vs 上下文 Embedding ---")
print(f"静态 Embedding: 同一个 Token 永远是同一个向量，所以同词不同义的相似度 = 1.0")
print(f"上下文 Embedding: 经过 Transformer 层后，同词不同义的相似度 < 1.0\n")

word_groups = [
    ("开", ["打开", "驾驶"]),
    ("热", ["温度高", "受欢迎"]),
    ("苹果", ["水果", "公司"]),
    ("花", ["花朵", "花费"]),
]

header = f'  {"词":<6s} {"含义1":<8s} {"含义2":<8s} {"静态相似度":<12s} {"上下文相似度":<12s} {"差异":<8s}'
print(header)
print("  " + "-" * (len(header) - 2))

for word, senses in word_groups:
    keys = [f"{word}({s})" for s in senses]
    valid_keys = [k for k in keys if k in emb_dict]
    if len(valid_keys) >= 2:
        ctx_sim = F.cosine_similarity(emb_dict[valid_keys[0]].unsqueeze(0), emb_dict[valid_keys[1]].unsqueeze(0)).item()
        diff = 1.0 - ctx_sim
        print(f"  {word:<6s} {senses[0]:<8s} {senses[1]:<8s} {1.0:<12.4f} {ctx_sim:<12.4f} {diff:<8.4f}")

print(f"\n--- 更有意思的发现 ---")
if "花(花朵)" in emb_dict and "苹果(水果)" in emb_dict and "花(花费)" in emb_dict:
    sim_cross = F.cosine_similarity(emb_dict["花(花朵)"].unsqueeze(0), emb_dict["苹果(水果)"].unsqueeze(0)).item()
    sim_same = F.cosine_similarity(emb_dict["花(花朵)"].unsqueeze(0), emb_dict["花(花费)"].unsqueeze(0)).item()
    print(f'  "花(花朵)" vs "苹果(水果)" = {sim_cross:.4f}  (不同词，同类含义)')
    print(f'  "花(花朵)" vs "花(花费)" = {sim_same:.4f}  (同一个词，不同含义)')
    if sim_cross > sim_same:
        print(f"  -> 模型认为'花(花朵)'跟'苹果(水果)'比跟'花(花费)'更近！")
        print(f"     这说明 Transformer 真正理解了语义，而不是仅仅看字面相同")

print(f"\n观察:")
print(f"  1. 静态 Embedding: 同词不同义相似度 = 1.0（完全无法区分）")
print(f"  2. 上下文 Embedding: 同词不同义相似度 < 1.0（Transformer 能区分不同含义）")
print(f"  3. '开(打开)' vs '开(驾驶)' 差异最大(0.53)，因为这两个含义几乎毫无关系")
print(f"  4. 语义相近的词（花-苹果）甚至比字面相同的词（花-花）更接近！")


=== 语境语义对比 ===

  "请帮我开门" -> 开(打开)
  "他开车上班" -> 开(驾驶)
  "这个苹果很甜" -> 苹果(水果)
  "苹果发布了新手机" -> 苹果(公司)
  "今天天气很热" -> 热(温度高)
  "他是个热门人物" -> 热(受欢迎)
  "这朵花很美" -> 花(花朵)
  "他花了很多钱" -> 花(花费)

--- 核心对比: 静态 Embedding vs 上下文 Embedding ---
静态 Embedding: 同一个 Token 永远是同一个向量，所以同词不同义的相似度 = 1.0
上下文 Embedding: 经过 Transformer 层后，同词不同义的相似度 < 1.0

  词      含义1      含义2      静态相似度        上下文相似度       差异      
  -----------------------------------------------------------
  开      打开       驾驶       1.0000       0.4707       0.5293  
  热      温度高      受欢迎      1.0000       0.5508       0.4492  
  苹果     水果       公司       1.0000       0.6797       0.3203  
  花      花朵       花费       1.0000       0.7383       0.2617  

--- 更有意思的发现 ---
  "花(花朵)" vs "苹果(水果)" = 0.7891  (不同词，同类含义)
  "花(花朵)" vs "花(花费)" = 0.7383  (同一个词，不同含义)
  -> 模型认为'花(花朵)'跟'苹果(水果)'比跟'花(花费)'更近！
     这说明 Transformer 真正理解了语义，而不是仅仅看字面相同

观察:
  1. 静态 Embedding: 同词不同义相似度 = 1.0（完全无法区分）
  2. 上下文 Embedding: 同词不同义相似度 < 1.0（Transformer 能区分不同含义）
  3. '开(打开)' vs '开

## 4. 变换：从 Token 到下一个 Token

现在我们有了 Token 的向量表示。接下来，这些向量要进入 **Transformer** 的层层变换，最终预测出下一个 Token。

整个流水线：

```
Token IDs → Embedding查表 → 24层Transformer Block → lm_head线性层 → Softmax概率 → 选出下一个Token
```

让我们用代码逐步拆解这个过程：

In [10]:
import torch.nn.functional as F

if 'model' not in dir():
    model = AutoModelForCausalLM.from_pretrained(tokenizer_id)
    print("模型已重新加载")

input_text = "你爱我，我爱你，蜜雪"
print(f"输入文本: '{input_text}'")

inputs = tokenizer(input_text, return_tensors="pt")
input_ids = inputs["input_ids"]
print(f"\n[Step 1] Token IDs: {input_ids.tolist()[0]}")

with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)
print(f"[Step 2] Embedding 形状: {embeddings.shape} -> (batch_size, seq_len, hidden_size)")

with torch.no_grad():
    transformer_outputs = model.model(input_ids)
    hidden_states = transformer_outputs.last_hidden_state
print(f"[Step 3] 最后一层 Hidden States 形状: {hidden_states.shape}")

last_token_hidden_state = hidden_states[:, -1, :]
print(f"[Step 4] 用于预测的最后一个 Token 的向量形状: {last_token_hidden_state.shape}")

with torch.no_grad():
    logits = model.lm_head(last_token_hidden_state)
print(f"[Step 5] Logits 形状: {logits.shape} -> (batch_size, vocab_size={len(tokenizer)})")

probs = F.softmax(logits, dim=-1)
next_token_id = torch.argmax(probs, dim=-1).item()
next_token_str = tokenizer.decode([next_token_id])

print(f"\n=== 预测结果 ===")
print(f"最高概率的 Token ID: {next_token_id}")
print(f"对应的文本是: '{next_token_str}'")
print(f"拼接后的完整句子: '{input_text + next_token_str}'")

输入文本: '你爱我，我爱你，蜜雪'

[Step 1] Token IDs: [95933, 121196, 3709, 115734, 3709, 98735, 97055]
[Step 2] Embedding 形状: torch.Size([1, 7, 1024]) -> (batch_size, seq_len, hidden_size)
[Step 3] 最后一层 Hidden States 形状: torch.Size([1, 7, 1024])
[Step 4] 用于预测的最后一个 Token 的向量形状: torch.Size([1, 1024])
[Step 5] Logits 形状: torch.Size([1, 248320]) -> (batch_size, vocab_size=248077)

=== 预测结果 ===
最高概率的 Token ID: 97260
对应的文本是: '冰'
拼接后的完整句子: '你爱我，我爱你，蜜雪冰'


### 自回归生成：一个接一个地“吐”出文本

上面只预测了一个 Token。但大模型的真正工作方式是**自回归**的：每预测出一个 Token，就把它拼回输入，再预测下一个，如此循环。

让我们把“你爱我，我爱你，蜜雪”接下去补完：

In [11]:
if 'model' not in dir():
    model = AutoModelForCausalLM.from_pretrained(tokenizer_id)

# Qwen3.5-0.8B 实测无法推理出魔性的歌词，有兴趣可以解开下边注释来试试
# tokenizer_id = "Qwen/Qwen3.5-9B"
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
# model = AutoModelForCausalLM.from_pretrained(tokenizer_id)

generated_text = "你爱我，我爱你，蜜雪"
input_ids = tokenizer.encode(generated_text, return_tensors="pt")
max_new_tokens = 20

print(f"输入: '{generated_text}'")
print(f"\n--- 自回归生成过程 ---")

for step in range(max_new_tokens):
    with torch.no_grad():
        outputs = model.model(input_ids)
        last_hidden = outputs.last_hidden_state[:, -1, :]
        logits = model.lm_head(last_hidden)
        probs = F.softmax(logits, dim=-1)
        next_id = torch.argmax(probs, dim=-1).item()
    
    next_str = tokenizer.decode([next_id])
    generated_text += next_str
    input_ids = torch.cat([input_ids, torch.tensor([[next_id]])], dim=1)
    
    print(f"  Step {step+1:>2d}: 预测 Token '{next_str}' (ID={next_id})  ->  当前文本: '{generated_text}'")
    
    if next_id == tokenizer.eos_token_id:
        print(f"  \u2714 生成结束符出现，停止生成。")
        break

print(f"\n=== 最终结果 ===")
print(f"'{generated_text}'")

输入: '你爱我，我爱你，蜜雪'

--- 自回归生成过程 ---
  Step  1: 预测 Token '冰' (ID=97260)  ->  当前文本: '你爱我，我爱你，蜜雪冰'
  Step  2: 预测 Token '城' (ID=96191)  ->  当前文本: '你爱我，我爱你，蜜雪冰城'
  Step  3: 预测 Token '，' (ID=3709)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，'
  Step  4: 预测 Token '你' (ID=95933)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你'
  Step  5: 预测 Token '爱我' (ID=121196)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我'
  Step  6: 预测 Token '，' (ID=3709)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，'
  Step  7: 预测 Token '我爱你' (ID=115734)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你'
  Step  8: 预测 Token '，' (ID=3709)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，'
  Step  9: 预测 Token '蜜' (ID=98735)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜'
  Step 10: 预测 Token '雪' (ID=97055)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪'
  Step 11: 预测 Token '冰' (ID=97260)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰'
  Step 12: 预测 Token '城' (ID=96191)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城'
  Step 13: 预测 Token '，' (ID=3709)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城，'
  Step 14: 预测 Token '你' (ID=95933)  ->  当前文本: '你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城，你'
  Step 1

## 5. 加速生成的秘密武器：KV Cache

在上一节的自回归生成中，我们每预测一个新 Token，都要把**整个序列**（包括之前已经处理过的所有 Token）重新送入模型计算。

这显然很浪费！因为之前 Token 的 Key 和 Value 向量其实已经算过了，没必要重复计算。

**KV Cache（键值缓存）** 的核心思想就是：**把每一层 Self-Attention 计算出的 Key 和 Value 向量缓存下来，下次只需要计算新 Token 的 Query，然后和缓存的 K、V 做注意力计算即可。**

```
无 KV Cache:  每一步都重新计算所有 Token 的 K、V
  Step 1: [T1]           → 计算 K1, V1
  Step 2: [T1, T2]       → 重新计算 K1, V1, K2, V2
  Step 3: [T1, T2, T3]   → 重新计算 K1, V1, K2, V2, K3, V3
  ...

有 KV Cache:  只计算新 Token 的 K、V，复用缓存的
  Step 1: [T1]           → 计算 K1, V1，缓存
  Step 2: [T2]           → 只计算 K2, V2，复用 K1, V1
  Step 3: [T3]           → 只计算 K3, V3，复用 K1-K2, V1-V2
  ...
```

这使得每一步的计算量从 O(seq_len²) 降低到 O(seq_len)，在长文本生成时加速效果非常显著。

> 📖 参考: [HuggingFace KV Cache 官方文档](https://huggingface.co/docs/transformers/v5.8.1/en/kv_cache)


In [14]:
import time

# ===== 自适应设备 =====
device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

# Qwen3.5-0.8B 实测无法推理出魔性的歌词，这里换个模型
tokenizer_id = "Qwen/Qwen3.5-9B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
model = AutoModelForCausalLM.from_pretrained(tokenizer_id).to(device).eval()

input_text = "你爱我，我爱你，蜜雪"
inputs = tokenizer(input_text, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
max_new_tokens = 20

# --- 无 KV Cache ---
# print("[无 KV Cache] 自回归生成...")
# start = time.time()
# generated_ids_no_cache = input_ids.clone()
# for step in range(max_new_tokens):
#     with torch.no_grad():
#         outputs = model.model(generated_ids_no_cache, use_cache=False)
#         last_hidden = outputs.last_hidden_state[:, -1, :]
#         logits = model.lm_head(last_hidden)
#         next_id = torch.argmax(logits, dim=-1).unsqueeze(0)
#     generated_ids_no_cache = torch.cat([generated_ids_no_cache, next_id], dim=1)
# no_cache_time = time.time() - start
# result_no_cache = tokenizer.decode(generated_ids_no_cache[0])
# print(f"  结果: {result_no_cache}")
# print(f"  耗时: {no_cache_time:.3f}s")

# --- 有 KV Cache ---
print(f"\n[有 KV Cache] 自回归生成...")
start = time.time()
generated_ids_cache = input_ids.clone()
past_key_values = None
for step in range(max_new_tokens):
    with torch.no_grad():
        if past_key_values is None:
            outputs = model.model(generated_ids_cache, past_key_values=past_key_values, use_cache=True)
        else:
            next_input = generated_ids_cache[:, -1:]
            outputs = model.model(next_input, past_key_values=past_key_values, use_cache=True)
        past_key_values = outputs.past_key_values
        last_hidden = outputs.last_hidden_state[:, -1, :]
        logits = model.lm_head(last_hidden)
        next_id = torch.argmax(logits, dim=-1).unsqueeze(0)
    generated_ids_cache = torch.cat([generated_ids_cache, next_id], dim=1)
cache_time = time.time() - start
result_cache = tokenizer.decode(generated_ids_cache[0])
print(f"  结果: {result_cache}")
print(f"  耗时: {cache_time:.3f}s")

print(f"\n=== 对比 ===")
print(f"  无 KV Cache: {no_cache_time:.3f}s")
print(f"  有 KV Cache: {cache_time:.3f}s")
print(f"  加速比: {no_cache_time/cache_time:.1f}x")
print(f"  结果一致: {result_no_cache == result_cache}")


Using device: mps


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]


[有 KV Cache] 自回归生成...
  结果: 你爱我，我爱你，蜜雪冰城甜蜜蜜。
蜜雪冰城，蜜雪冰城，甜蜜蜜，甜蜜
  耗时: 10.426s

=== 对比 ===
  无 KV Cache: 7.479s
  有 KV Cache: 10.426s
  加速比: 0.7x
  结果一致: True


### KV Cache 的代价与优化

KV Cache 虽然加速了生成，但也有代价：**显存占用**。每缓存一个 Token 的 K、V 向量，需要消耗 `2 × 层数 × 隐藏维度` 的存储空间。

对于长文本生成（如 100K Token 的上下文），KV Cache 可能占用数 GB 甚至数十 GB 的显存。
因此，HuggingFace Transformers 提供了多种 Cache 优化策略（参考 [官方文档](https://huggingface.co/docs/transformers/v5.8.1/en/kv_cache)）：

| Cache 类型 | 特点 | 适用场景 |
| --- | --- | --- |
| **DynamicCache** | 默认，动态增长 | 通用场景 |
| **StaticCache** | 预分配固定大小，支持 torch.compile | 追求极致延迟 |
| **QuantizedCache** | 将 K、V 量化为低精度（如 int4） | 显存受限场景 |
| **OffloadedCache** | 将 K、V 卸载到 CPU | GPU 显存不足时 |

> 💡 在 CPU 上运行时，KV Cache 的加速效果可能不明显（因为瓶颈在计算而非内存带宽）。但在 GPU 上，加速效果非常显著。


## 结语

让我们回顾一下这句话的完整旅程：

> **“你爱我，我爱你，蜜雪”** → 分词为7个Token → 每个Token变成1024维向量 → 经过24层Transformer变换 → 从248K个候选中选出**“冰”** → 拼接后继续预测...

这就是大模型“一个字一个字”生成文本的真正原理：

```
文本 → Tokenize → Token IDs → Embedding → Transformer x N → Logits → Softmax → 下一个Token → 循环
```

而 KV Cache 让这个循环在每一步避免重复计算，大幅加速了生成过程。

但文本只是世界的一部分。图片、视频、声音……这些模态的信息又是如何变成 Token 的呢？

这些问题，我们将在 **[多模态 Token 探秘](./multimodal_tokens.ipynb)** 中一一揭晓。